In [1]:
import os

import numpy as np
import pandas as pd
import pyreadr

from trunx.config import threepg_data_folder

In [4]:
data = pyreadr.read_r("../models/r3PG/vignettes_build/vignette_data/solling.rda")
data.keys()

odict_keys(['site_solling', 'species_solling', 'climate_solling', 'thinn_solling', 'param_solling', 'error_solling', 'observ_solling', 'observ_solling_mat'])

In [ ]:
# Exploring the structure of the data in r3PG vignette_data 
# import rpy2.robjects as ro

# # Load the RDA file
# ro.r["load"]("../models/r3PG/vignettes_build/vignette_data/grid_input.rda")

# # Get the loaded objects
# loaded_objects = list(ro.r["ls"]())
# print(f"Objects in file: {loaded_objects}")

# # Assuming the first object is your MCMC output
# mcmc_obj = ro.r[loaded_objects[3]]
# print(f"Object class: {mcmc_obj.rclass}")
# print("Object structure:")
# print(ro.r["str"](mcmc_obj))

# Try different extraction methods based on the object type

In [3]:
climate = pd.DataFrame(data["climate_solling"])
climate["idx"] = np.arange(0, len(climate))

obv = pd.DataFrame(data["observ_solling"])
obv["month"] = pd.to_datetime(obv["date"]).dt.month
obv["year"] = pd.to_datetime(obv["date"]).dt.year

obv = obv.merge(climate, on=["month", "year"])[
    [
        "idx",
        "month",
        "year",
        "date",
        "biom_stem",
        "biom_foliage",
        "biom_root",
        "basal_area",
        "stems_n",
        "dbh",
        "height",
    ]
]

obv = obv.rename(
    columns={
        "dbh": "DBH",
        "biom_stem": "WS",
        "biom_foliage": "WF",
        "biom_root": "WR",
        "basal_area": "BA",
        "stems_n": "N",
        "height": "Height",
    }
)

In [4]:
param_df = pd.DataFrame(data["param_solling"])

params_bounds = {}
for _, row in param_df.iterrows():
    if pd.notna(row["min"]) and pd.notna(row["max"]):
        params_bounds[row["param_name"]] = (row["min"], row["max"])

In [5]:
error_solling = data["error_solling"]
error_solling["param_name"] = [
    "err_" + metric for metric in ["BA", "DBH", "Height", "WS", "WR", "WF"]
]

error_solling

,param_name,default,min,max
0,err_BA,6.0,0.0,30.0
1,err_DBH,3.0,0.0,15.0
2,err_Height,3.0,0.0,10.0
3,err_WS,5.0,0.0,30.0
4,err_WR,1.0,0.0,15.0
5,err_WF,1.0,0.0,10.0


In [6]:
param_default = pd.read_excel(os.path.join(threepg_data_folder, "data.default.xlsx"))

params = pd.DataFrame(data["param_solling"])
params = params.rename(columns={"param_name": "parameter", "default": "piab"})
params = params[["parameter", "piab"]]
params["piab"] = params["piab"].fillna(param_default["default"])

params.head()

,parameter,piab
0,pFS2,0.363569
1,pFS20,0.139412
2,aWS,0.069847
3,nWS,2.432750
4,pRx,0.352658


In [7]:
with pd.ExcelWriter(
    os.path.join(threepg_data_folder, "solling_data.xlsx"), engine="openpyxl"
) as writer:
    data["climate_solling"].to_excel(writer, sheet_name="climate", index=False)
    data["site_solling"].to_excel(writer, sheet_name="site", index=False)
    data["species_solling"].to_excel(writer, sheet_name="species", index=False)
    params.to_excel(writer, sheet_name="parameters", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="thinning", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="sizeDist", index=False)
    obv.to_excel(writer, sheet_name="observed", index=False)
    data["param_solling"].to_excel(writer, sheet_name="param_bound", index=False)
    error_solling.to_excel(writer, sheet_name="error_param", index=False)

# Davos data processing

In [8]:
import polars as pl

davos_df = pl.read_csv(os.path.join(threepg_data_folder, "davos_meteo_data_site_id_36.csv"))

davos_df = davos_df.with_columns(pl.col("ts").str.to_datetime("%Y-%m-%d"))
davos_df = davos_df.with_columns(
    [pl.col("ts").dt.month().alias("month"), pl.col("ts").dt.year().alias("year")]
)

davos_df = davos_df.rename(
    {
        "tas": "tmp_ave",
        "tasmax": "tmp_max",
        "tasmin": "tmp_min",
        "pr": "prcp",
        "gh": "srad",
        "vpd": "vpd_day",
    }
)

davos_df = davos_df.select(
    ["ts", "month", "year", "tmp_ave", "tmp_max", "tmp_min", "prcp", "srad", "vpd_day"]
)

davos_df = (
    davos_df.group_by(["month", "year"])
    .agg(
        pl.col("tmp_ave").mean().alias("tmp_ave"),
        pl.col("tmp_max").mean().alias("tmp_max"),
        pl.col("tmp_min").mean().alias("tmp_min"),
        pl.col("prcp").sum().alias("prcp"),
        pl.col("srad").mean().alias("srad"),
        pl.col("vpd_day").mean().alias("vpd_day"),
        (pl.col("tmp_ave") < 0.0).sum().alias("frost_days"),
    )
    .sort(["year", "month"])
)

davos_df

month,year,tmp_ave,tmp_max,tmp_min,prcp,srad,vpd_day,frost_days
i8,i32,f64,f64,f64,f64,f64,f64,u32
1,1981,-9.03871,-4.167742,-13.764516,126.60022,64.787097,0.077423,31
2,1981,-8.328572,-1.242857,-13.425,26.500122,127.932144,0.123406,28
3,1981,-0.329032,4.658065,-4.422581,83.100586,155.206453,0.192981,16
4,1981,2.673333,8.47,-2.29,17.900024,242.380003,0.285408,7
5,1981,5.519355,10.390323,0.793548,82.900146,215.154841,0.34081,3
…,…,…,…,…,…,…,…,…
8,2025,12.919355,18.196774,7.758065,154.500365,220.53871,0.472621,0
9,2025,8.953333,13.81,4.673333,113.500367,162.080002,0.299979,0
10,2025,3.822581,9.093549,-0.383871,54.700561,126.283871,0.212701,3


In [9]:
import pandas as pd
from docx import Document

param_default = pd.read_excel(os.path.join(threepg_data_folder, "data.default.xlsx"))


def docx_tables_to_dfs(file_path):
    """Extract all tables from a .docx file and return a list of DataFrames."""
    doc = Document(file_path)
    all_dfs = []

    for table in doc.tables:
        data = []
        for row in table.rows:
            row_data = [cell.text.strip() for cell in row.cells]
            data.append(row_data)

        # Convert to DataFrame
        if data:
            # Assume first row is header
            df = pd.DataFrame(data[1:], columns=data[0])
            all_dfs.append(df)

    return all_dfs


# Use the function
file_path = os.path.join(threepg_data_folder, "gcb15011-sup-0001-supinfo-2.docx")
dfs = docx_tables_to_dfs(file_path)

# Species: P. abies
param_df = dfs[2]
param_df.head()
parameter_df = pd.DataFrame(param_df.values[1:], columns=param_df.values[0])[
    ["Parameter", "50.00%"]
]
parameter_df.rename(columns={"50.00%": "piab", "Parameter": "parameter"}, inplace=True)

parameter_df = pd.merge(param_default, parameter_df, on="parameter", how="left")
parameter_df["piab"] = parameter_df["piab"].fillna(parameter_df["default"])

parameter_df = parameter_df[["parameter", "piab"]]

ModuleNotFoundError: No module named 'exceptions'

In [ ]:
# Site Data

site_df = {
    "latitude": 46.802,
    "altitude": 1650,
    "soil_class": 3,
    "asw_i": 999,
    "asw_min": 0,
    "asw_max": 106,
    "from": "1981-01",
    "to": "2025-12",
}

species_df = {
    "species": "piab",
    "planted": "1800-01",
    "fertility": 0.5,
    "stem_n": 830,
}

In [ ]:
with pd.ExcelWriter(
    os.path.join(threepg_data_folder, "davos_data.xlsx"), engine="openpyxl"
) as writer:
    davos_df.to_pandas().to_excel(writer, sheet_name="climate", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="thinning", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="sizeDist", index=False)
    parameter_df.to_excel(writer, sheet_name="parameters", index=False)

In [ ]:
import pandas as pd

davos_climate_df = pd.read_csv(os.path.join(threepg_data_folder, "Davos_climate_monthly.csv"))
davos_gpp = pd.read_csv(os.path.join(threepg_data_folder, "Davos_GoSIF_GPP_monthly.csv"))

davos_gpp = davos_gpp.rename(columns={"gpp_43650": "GPP"})
davos_gpp["GPP"] = davos_gpp["GPP"] / 12.011
davos_gpp = davos_gpp[davos_gpp["year"] >= 2013]

parameters_dict = {
    # Growth and allocation parameters
    "pFS2": 0.8297,  # Foliage:stem ratio at D=2cm
    "pFS20": 0.1480,  # Foliage:stem ratio at D=20cm
    "aWS": 0.1330,  # Stem mass-diameter allometry constant
    "nWS": 2.3050,  # Stem mass-diameter allometry exponent
    "pRx": 0.7151,  # Max fraction of NPP to roots
    "pRn": 0.0824,  # Min fraction of NPP to roots
    # Litterfall and turnover
    "gammaF1": 0.0008,  # Max litterfall rate (1/month)
    "gammaF0": 0.0010,  # Litterfall rate at t=0
    "tgammaF": 60,  # Age at which litterfall has median value (months)
    "gammaR": 0.0000,  # Root turnover rate
    "leafgrow": 0.0000,  # (evergreen: 0)
    "leaffall": 0.0000,  # (evergreen: 0)
    # Temperature response
    "Tmin": 2.4727,  # Min temperature for growth (deg C)
    "Topt": 24.9494,  # Optimum temperature for growth (deg C)
    "Tmax": 30.1378,  # Max temperature for growth (deg C)
    "kF": 1.0000,  # Days production lost per frost day
    # Water stress
    "SWconst": 0.7000,  # Moisture ratio deficit for fq=0.5
    "SWpower": 9.0000,  # Power of moisture ratio deficit
    # CO2 response
    "fCalpha700": 1.0040,  # CO2 assimilation enhancement at 700 ppm
    "fCg700": 0.9584,  # Canopy conductance enhancement at 700 ppm
    # Nutrition
    "m0": 0.0000,  # m when FR=0
    "fN0": 0.6000,  # Modifier for nutrition when FR=0
    "fNn": 1.0000,  # Power of nutrition modifier
    # Age modifiers
    "MaxAge": 400,  # Max stand age for age modifier
    "nAge": 4.0000,  # Power of relative age in age modifier
    "rAge": 0.9500,  # Relative age modifier at MaxAge
    # Mortality
    "gammaN1": 0.0000,  # Max density-independent mortality
    "gammaN0": 0.0000,  # Seedling mortality rate
    "tgammaN": 0.0000,  # Age at max density-independent mortality
    "ngammaN": 1.0000,  # Shape of mortality response
    # Self-thinning
    "wSx1000": 376.0843,  # Max stem mass at 1000 stems ha-1
    "thinPower": 1.8740,  # Power in self-thinning rule
    # Mortality fractions
    "mF": 0.4880,  # Leaf mortality fraction
    "mR": 0.4360,  # Root mortality fraction
    "mS": 0.4370,  # Stem mortality fraction
    # Specific leaf area
    "SLA0": 8.7100,  # Specific leaf area at age 0 (m2/kg)
    "SLA1": 3.8500,  # Specific leaf area at MaxAge (m2/kg)
    "tSLA": 25.1000,  # Age at which SLA = (SLA0+SLA1)/2
    # Canopy and radiation
    "k": 0.6378,  # Extinction coefficient for APAR
    "fullCanAge": 3.0000,  # Age at full canopy closure
    "MaxIntcptn": 0.2237,  # Max proportion of rainfall intercepted
    "LAImaxIntcptn": 3.0000,  # LAI for max rainfall interception
    "cVPD": 5.0000,  # VPD at which stomata close
    "alphaCx": 0.0700,  # *OVERRIDDEN* Max canopy quantum efficiency (gDM/MJ)
    "Y": 0.4700,  # Ratio NPP/GPP (construction respiration)
    # Stomatal conductance
    "MinCond": 0.0000,  # Minimum stomatal conductance
    "MaxCond": 0.0246,  # Maximum stomatal conductance
    "LAIgcx": 3.3300,  # LAI for max conductance
    "CoeffCond": 0.0896,  # Conductance-VPD coefficient
    "BLcond": 0.2000,  # Canopy boundary layer conductance
    "RGcGw": 0.6600,  # Ratio of stem to leaf conductance
    # Carbon discrimination
    "D13CTissueDif": 2.0000,  # Leaf-atmosphere 13C discrimination
    "aFracDiffu": 4.4000,  # Mesophyll conductance parameter
    "bFracRubi": 27.0000,  # Rubisco carboxylation parameter
    # Branch and bark
    "fracBB0": 0.0000,  # Branch+bark fraction at age 0
    "fracBB1": 0.0000,  # Branch+bark fraction at max age
    "tBB": 0.0000,  # Age at which fracBB=(fracBB0+fracBB1)/2
    # Density
    "rhoMin": 0.4000,  # Min basic density (t/m3)
    "rhoMax": 0.4000,  # Max basic density (t/m3)
    "tRho": 1.0000,  # Age at which density = (rhoMin+rhoMax)/2
    # Crown and allometry
    "crownshape": 3.0000,  # Crown shape (3=ellipsoid)
    "aH": 37.7300,  # Height-diameter allometry constant
    "nHB": 17.8500,  # Height-diameter: stem mass power
    "nHC": 0.0064,  # Height-diameter: competition power
    "aV": 0.000115,  # Volume allometry constant
    "nVB": 2.3100,  # Volume allometry: stem mass power
    "nVH": 0.3300,  # Volume allometry: height power
    "nVBH": 0.0000,  # (No description provided)
    # Crown area
    "aK": 0.6300,  # Crown area allometry constant
    "nKB": 0.6400,  # Crown area: stem mass power
    "nKH": 0.0000,  # (No description provided)
    "nKC": -0.0690,  # (No description provided)
    "nKrh": 0.0670,  # (No description provided)
    # Leaf area allometry
    "aHL": 35.1800,  # Leaf area-height allometry constant
    "nHLB": 27.1800,  # (No description provided)
    "nHLL": 0.0000,  # (No description provided)
    "nHLC": -0.0050,  # (No description provided)
    "nHLrh": 0.0000,  # (No description provided)
    # Radiation conversion
    "Qa": -90.000,  # Intercept of net radiation vs. solar radiation
    "Qb": 0.8000,  # Slope of net radiation vs. solar radiation
    "gDM_mol": 24.0000,  # Conversion: gDM per mol CO2
    "molPAR_MJ": 2.3000,  # Conversion: mol PAR per MJ
}

params_df = pd.DataFrame(
    {"parameter": list(parameters_dict.keys()), "Picea abies": list(parameters_dict.values())}
)

species_df = pd.DataFrame(
    {
        "species": ["Picea abies"],
        "planted": ["1919-01"],
        "fertility": [0.5],
        "stems_n": [806],  # trees per hectare
        "biom_stem": [162],  # tDM ha-1
        "biom_root": [33],  # tDM ha-1
        "biom_foliage": [13],  # tDM ha-1
    }
)

site_df = pd.DataFrame(
    {
        "latitude": [46.813],
        "altitude": [1650],
        "soil_class": [3],
        "asw_i": [130],  # initial available soil water (mm)
        "asw_min": [0],  # minimum available soil water (mm)
        "asw_max": [150],  # maximum available soil water (mm)
        "from": ["2013-01"],
        "to": ["2023-01"],
    }
)


with pd.ExcelWriter(
    os.path.join(threepg_data_folder, "Davos_data_GPP.xlsx"), engine="openpyxl"
) as writer:
    davos_climate_df.to_excel(writer, sheet_name="climate", index=False)
    site_df.to_excel(writer, sheet_name="site", index=False)
    species_df.to_excel(writer, sheet_name="species", index=False)
    params_df.to_excel(writer, sheet_name="parameters", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="thinning", index=False)
    pd.DataFrame().to_excel(writer, sheet_name="sizeDist", index=False)
    davos_gpp.to_excel(writer, sheet_name="full_observed", index=False)